In [1]:
#Importng libraries

import os
import json

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("AI AGENT ENVIRONMENT READY!!")

AI AGENT ENVIRONMENT READY!!


In [2]:
#first agent tool
def multiply(a,b):
    result=a*b
    return result

In [3]:
print(multiply(25, 16))

400


In [4]:
#Telling the model which tools are available

tools = [
    {
        "type": "function",
        "name": "multiply",
        "description": "Multiply two numbers and return the result.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "number",
                    "description": "The first number"
                },
                "b": {
                    "type": "number",
                    "description": "The second number"
                }
            },
            "required": ["a", "b"],
            "additionalProperties": False
        },
        "strict": True
    }
]

In [5]:
#ASking the model to use the tools
response=client.responses.create(
    model="gpt-4.1-mini",
    instructions="You are a helpful assistant. Use the multiply tool for multiplication questions.",
    input="What is 25 multiplied by 16?",
    tools=tools,
    tool_choice="required"
    
)

for item in response.output:
    print(item)

ResponseFunctionToolCall(arguments='{"a":25,"b":16}', call_id='call_FzBX91u3S7zFEWTegamIZt1f', name='multiply', type='function_call', id='fc_09221c1d6ae4e1e0006aeeb79a817887d28f1b55c145ea9509', async_=None, caller=None, namespace=None, status='completed')


In [6]:
#Now connecting the model's reques to our actucal python function

for item in response.output:

    if item.type =="function_call":

        if item.name=="multiply":
            arguments=json.loads(item.arguments)

            result=multiply(**arguments)

            print("Tool Result:",result)

Tool Result: 400


In [7]:
tool_response = client.responses.create(
    model="gpt-4.1-mini",
    previous_response_id=response.id,
    input=[
        {
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": str(result)
        }
    ]
)

print(tool_response.output_text)

25 multiplied by 16 is 400.


In [8]:
#putting everything in a single function
def calculator_agent(question):

    response = client.responses.create(
        model="gpt-4.1-mini",
        instructions="Use the multiply tool for multiplication questions.",
        input=question,
        tools=tools,
        tool_choice="required"
    )

    for item in response.output:

        if item.type == "function_call":

            if item.name == "multiply":

                arguments = json.loads(item.arguments)

                result = multiply(**arguments)

                tool_response = client.responses.create(
                    model="gpt-4.1-mini",
                    previous_response_id=response.id,
                    input=[
                        {
                            "type": "function_call_output",
                            "call_id": item.call_id,
                            "output": str(result)
                        }
                    ]
                )

                return tool_response.output_text

    return "The model did not request the multiply tool."

In [9]:
print(calculator_agent("What is 125 multiplied by 8?"))

125 multiplied by 8 is 1000.
